In [16]:
from ucimlrepo import fetch_ucirepo
import plotly.express as px
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import GridSearchCV


In [9]:
warnings.simplefilter("ignore", category=ConvergenceWarning)

In [4]:
wine_quality = fetch_ucirepo(name='Wine Quality')
wine_quality_data = wine_quality['data']['original']

In [5]:
seed = 42 #if it must be 42 :P

wine_X = wine_quality_data.drop(columns = ['quality']) # drop quality
wine_X['color'] = (wine_X['color'] == "red").astype(float) # make color a number instead of a string
scaler = StandardScaler()
wine_X = scaler.fit_transform(wine_X) # scaling values
num_features = wine_X.shape[1]

wine_y = wine_quality_data['quality']
min_quality = wine_y.min()
num_classes = wine_y.max() - min_quality + 1
wine_y -= min_quality

wine_X_train_val, wine_X_test, wine_y_train_val, wine_y_test = train_test_split(
    wine_X, wine_y, test_size=0.2, random_state=42
)

wine_X_tr, wine_X_val, wine_y_tr, wine_y_val = train_test_split(
    wine_X_train_val, wine_y_train_val, test_size=0.25, random_state=42
)

In [13]:
penalty = [None, 'l1', 'l2', 'elasticnet']
# no liblinear because it's not multiclass and the data set isn't even small
solvers = ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga']
cs = [0.001, 0.01, 1, 1.001, 1.01, 1.1, 2, 5]

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=ConvergenceWarning)
    warnings.simplefilter("ignore", category=UserWarning)
    for solv in solvers:
        l1rat = 0
        for pen in penalty:
            match pen:
                case 'elasticnet':
                    l1rat = 0.5
                case 'l1':
                    l1rat = 1
            try:
                max_iters = 0
                max_val_accuracy = 0
                max_classifier = LogisticRegression(solver=solv, penalty=pen, max_iter = 1, random_state = seed)
                for iters in range(10, 200, 10):
                    classifier = LogisticRegression(solver=solv, penalty=pen, max_iter = iters, l1_ratio=l1rat, random_state = seed);
                    classifier.fit(wine_X_tr, wine_y_tr);
                    if(100*accuracy_score(wine_y_val, classifier.predict(wine_X_val)) > max_val_accuracy):
                        max_iters = iters
                        max_val_accuracy = 100*accuracy_score(wine_y_val, classifier.predict(wine_X_val))
                        max_classifier = classifier 
                print(f'Params: solver: {solv}, penalty: {pen}, max_iters: {max_iters}')
                print(f'Training accuracy: {100*accuracy_score(wine_y_tr, max_classifier.predict(wine_X_tr)):.2f}%' )
                print(f'Validation accuracy: {100*accuracy_score(wine_y_val, max_classifier.predict(wine_X_val)):.2f}%\n' )
            except ValueError:
                continue


#...i've come to the conclusion I'm using saga with l1 penalty 


Params: solver: lbfgs, penalty: None, max_iters: 90
Training accuracy: 54.94%
Validation accuracy: 55.38%

Params: solver: lbfgs, penalty: l2, max_iters: 80
Training accuracy: 54.68%
Validation accuracy: 55.31%

Params: solver: newton-cg, penalty: None, max_iters: 20
Training accuracy: 54.89%
Validation accuracy: 55.00%

Params: solver: newton-cg, penalty: l2, max_iters: 10
Training accuracy: 54.66%
Validation accuracy: 55.15%

Params: solver: newton-cholesky, penalty: None, max_iters: 10
Training accuracy: 54.89%
Validation accuracy: 55.00%

Params: solver: newton-cholesky, penalty: l2, max_iters: 10
Training accuracy: 54.66%
Validation accuracy: 55.15%

Params: solver: sag, penalty: None, max_iters: 20
Training accuracy: 54.58%
Validation accuracy: 55.15%

Params: solver: sag, penalty: l2, max_iters: 20
Training accuracy: 54.61%
Validation accuracy: 55.23%

Params: solver: saga, penalty: None, max_iters: 40
Training accuracy: 54.61%
Validation accuracy: 55.15%

Params: solver: saga, 

In [ ]:
ratio = 0
max_val_accuracy = 0
max_classifier = LogisticRegression(max_iter = 1, random_state = seed)

for i in range(0, 101):
    classifier = LogisticRegression(solver='saga', penalty='elasticnet', l1_ratio = i/100, max_iter = 100, random_state = seed)
    classifier.fit(wine_X_tr, wine_y_tr);
    if(100*accuracy_score(wine_y_val, classifier.predict(wine_X_val)) >= max_val_accuracy):
        ratio = i
        max_val_accuracy = 100*accuracy_score(wine_y_val, classifier.predict(wine_X_val))
        max_classifier = classifier 

print(f'Params: solver: {solv}, penalty: {pen}, i: {ratio}')
print(f'Training accuracy: {100*accuracy_score(wine_y_tr, max_classifier.predict(wine_X_tr)):.2f}%' )
print(f'Validation accuracy: {100*accuracy_score(wine_y_val, max_classifier.predict(wine_X_val)):.2f}%\n' )

#conclusion: absolutely saga L1 at 100 iterations

Params: solver: saga, penalty: elasticnet, i: 95
Training accuracy: 54.76%
Validation accuracy: 55.46%



In [14]:
max_c = 0
max_val_accuracy = 0
max_classifier = LogisticRegression(max_iter = 1, random_state = seed)

for c in cs:
    classifier = LogisticRegression(solver='saga', penalty='l1', C = c, max_iter = 100, random_state = seed)
    classifier.fit(wine_X_tr, wine_y_tr);
    if(100*accuracy_score(wine_y_val, classifier.predict(wine_X_val)) >= max_val_accuracy):
        max_c = c
        max_val_accuracy = 100*accuracy_score(wine_y_val, classifier.predict(wine_X_val))
        max_classifier = classifier 

print(f'Params: solver: {solv}, penalty: {pen}, c: {max_c}')
print(f'Training accuracy: {100*accuracy_score(wine_y_tr, max_classifier.predict(wine_X_tr)):.2f}%' )
print(f'Validation accuracy: {100*accuracy_score(wine_y_val, max_classifier.predict(wine_X_val)):.2f}%\n' )

#conclusion: absolutely saga L1 at 100 iterations

Params: solver: saga, penalty: elasticnet, c: 1.01
Training accuracy: 54.76%
Validation accuracy: 55.38%



In [ ]:
# penalty = [None, 'l1', 'l2', 'elasticnet']
# # no liblinear because it's not multiclass and the data set isn't even small
# solvers = ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga']
# cs = [0.001, 0.01, 1, 1.001, 1.01, 1.1, 2, 5]

classifier = LogisticRegression(random_state=seed)

param_grid = {
    'solver': ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
    'C': [0.001, 0.01, 1, 1.001, 1.01, 1.05, 1.1, 1.5, 2, 5],
    'penalty': [None, 'l1', 'l2', 'elasticnet'],
    'l1_ratio': [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]
}

# Step 4: Instantiate GridSearchCV
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,            # Use 5-fold cross-validation
    scoring='accuracy', # Metric to optimize
    verbose=1,       # Output progress
    n_jobs=-1        # Use all available cores
)


In [ ]:
# # Create a figure with only one subplot
# figure, axes = plt.subplots(1, figsize=(6, 6))

# ### YOUR CODE STARTS HERE ###

# errors_tr = []
# errors_te = []

# k = [1, 2, 5, 10, 50, 100, 110]
# for i in range(len(k)):
#     knn_classifier = KNeighborsClassifier(n_neighbors=k[i])
#     knn_classifier.fit(peng_X_tr, peng_y_tr)
#     y_pred = knn_classifier.predict(peng_X_tr)
#     accuracy = accuracy_score(peng_y_tr, y_pred)
#     errors_tr.append(1-accuracy)
#     y_pred = knn_classifier.predict(peng_X_te)
#     accuracy = accuracy_score(peng_y_te, y_pred)
#     errors_te.append(1-accuracy)

# axes.semilogx(k, errors_tr, c='r', label = "training")
# axes.semilogx(k, errors_te, c = 'g', label = "testing")
# axes.set_xlabel('k')
# axes.set_ylabel('error')
# axes.legend()

# ### YOUR CODE ENDS HERE ###